In [39]:
# Import required libraries
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError
import os
from dotenv import load_dotenv
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

# Load environment variables
load_dotenv()

# Define period
period = '202603'
delete_placeholder = '33'

print("Libraries imported successfully!")


Libraries imported successfully!


In [40]:
# Database connection configuration to SIMPEG
DB_HOST = os.getenv('DB_HOST_SIMPEG', 'localhost')
DB_PORT = os.getenv('DB_PORT_SIMPEG', '5432')  # PostgreSQL default port
DB_NAME = os.getenv('DB_DATABASE_SIMPEG', 'your_database_name')
DB_USER = os.getenv('DB_USERNAME_SIMPEG', 'your_username')
DB_PASSWORD = os.getenv('DB_PASSWORD_SIMPEG', 'your_password')

# Create connection string for PostgreSQL
connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

print(f"Connecting to database: {DB_NAME} on {DB_HOST}:{DB_PORT}")
print(f"User: {DB_USER}")

# Test connection
try:
    engine_simpeg = create_engine(connection_string, echo=False)
    with engine_simpeg.connect() as connection:
        result = connection.execute(text("SELECT 1 as test"))
        print("✅ Database connection successful!")
        print(f"Connection test result: {result.fetchone()[0]}")
except SQLAlchemyError as e:
    print(f"❌ Database connection failed: {e}")
    print("Please check your database credentials in the .env file")


Connecting to database: simpeg_jabar on 10.110.32.121:5432
User: postgres
✅ Database connection successful!
Connection test result: 1


In [41]:
def load_data_from_sql(query, engine):
    """
    Load data from PostgreSQL database into a pandas DataFrame.
    
    Parameters:
    query (str): SQL query to execute
    engine: SQLAlchemy engine object
    
    Returns:
    pandas.DataFrame: Data from the query
    """
    connection = None
    try:
        # Create a new connection and rollback any pending transaction
        connection = engine.connect()
        
        # Rollback any pending transaction to ensure clean state
        try:
            connection.rollback()
        except:
            pass  # If no transaction to rollback, ignore
        
        # Execute the query
        df = pd.read_sql(query, connection)
        print(f"✅ Data loaded successfully! Shape: {df.shape}")
        
        connection.close()
        return df
    except Exception as e:
        # Ensure connection is closed on error
        if connection:
            try:
                connection.rollback()
            except:
                pass
            try:
                connection.close()
            except:
                pass
        print(f"❌ Error loading data: {e}")
        return None

def get_table_info(table_name, engine):
    """
    Get basic information about a table.
    
    Parameters:
    table_name (str): Name of the table
    engine: SQLAlchemy engine object
    """
    try:
        # Get table structure using PostgreSQL information_schema
        structure_query = f"""
            SELECT 
                column_name,
                data_type,
                character_maximum_length,
                is_nullable,
                column_default
            FROM information_schema.columns
            WHERE table_name = '{table_name}'
            ORDER BY ordinal_position
        """
        structure = pd.read_sql(structure_query, engine)
        
        # Get row count
        count_query = f"SELECT COUNT(*) as row_count FROM {table_name}"
        count_result = pd.read_sql(count_query, engine)
        
        print(f"📊 Table: {table_name}")
        print(f"Rows: {count_result['row_count'].iloc[0]}")
        print(f"Columns: {len(structure)}")
        print("\nColumn Information:")
        print(structure)
        
        return structure
    except SQLAlchemyError as e:
        print(f"❌ Error getting table info: {e}")
        return None

def list_tables(engine):
    """
    List all tables in the database.
    
    Parameters:
    engine: SQLAlchemy engine object
    """
    try:
        # Use PostgreSQL information_schema to list tables
        query = """
            SELECT table_name 
            FROM information_schema.tables 
            WHERE table_schema = 'public'
            ORDER BY table_name
        """
        tables = pd.read_sql(query, engine)
        print("📋 Available tables:")
        for table in tables['table_name']:
            print(f"  - {table}")
        return tables
    except SQLAlchemyError as e:
        print(f"❌ Error listing tables: {e}")
        return None

print("Data loading functions defined successfully!")

def execute_update(query, params, engine, description=""):
    """
    Execute an UPDATE query and return success status.
    
    Parameters:
    query (str): SQL UPDATE query
    params (dict): Parameters for the query
    engine: SQLAlchemy engine
    description (str): Description for logging
    
    Returns:
    dict: {'success': bool, 'rows_affected': int, 'error': str}
    """
    connection = None
    try:
        connection = engine.connect()
        
        # Rollback any pending transaction
        try:
            connection.rollback()
        except:
            pass
        
        # Execute the UPDATE with parameters
        result = connection.execute(text(query), params)
        rows_affected = result.rowcount
        
        # Commit the transaction
        connection.commit()
        connection.close()
        
        return {
            'success': True,
            'rows_affected': rows_affected,
            'error': None,
            'description': description
        }
    except Exception as e:
        # Rollback on error
        if connection:
            try:
                connection.rollback()
            except:
                pass
            try:
                connection.close()
            except:
                pass
        
        return {
            'success': False,
            'rows_affected': 0,
            'error': str(e),
            'description': description
        }



Data loading functions defined successfully!


In [42]:
# Load data from pkl

df_vpd = pd.read_pickle('df_vpd.pkl')

# Remove rows where peg_status = false
df_vpd = df_vpd[df_vpd['peg_status'] == True]

df_vpd.head()

,peg_id,peg_nip,peg_nama,peg_lahir_tanggal,peg_usia,peg_jenis_kelamin,peg_status,peg_ketstatus,peg_umur_pensiun,peg_tmt_pensiun,...,is_gtk,nip_atasan,nama_atasan,nip_atasan_bayangan,nama_atasan_bayangan,unit_kerja_nama_full,is_nakes,is_atasan_bayangan,is_atasan_tugas_tambahan,atasan_kinerja
0,197407232000121002,197407232000121002,DUDUNG JALALUDIN,1974-07-23,51,L,True,None,60,2034-08-01,...,True,198003242006042015,NOOR RAHMAWATI,197912162003121001,IIM IMANSYAH,SMAN 1 CIAMIS KABUPATEN CIAMIS|SATUAN PENDIDIK...,False,True,True,None
1,198209062010012012,198209062010012012,KRISNA SULISTYA,1982-09-06,43,P,True,None,60,2042-10-01,...,True,197906052009011012,YAYAN SURYANA,196702191994121002,KARNITA,SMAN 13 BANDUNG KOTA BANDUNG|SATUAN PENDIDIKAN...,False,True,False,None
2,196806141997031004,196806141997031004,ARIEF NADJEMUDIN,1968-06-14,57,L,True,None,60,2028-07-01,...,False,197306241998031008,YOGI GAUTAMA JAELANI,197306241998031008,YOGI GAUTAMA JAELANI,BIRO HUKUM DAN HAK ASASI MANUSIA|ASISTEN PEMER...,False,True,False,None
3,199006122025212105,199006122025212105,NURHANIFAH,1990-06-12,35,P,True,None,58,2048-07-01,...,True,196708211990031007,SUGENG PRAYITNO,None,None,SMAN 1 SINDANG KABUPATEN INDRAMAYU|SATUAN PEND...,False,False,True,None
4,197108202025211030,197108202025211030,JAJANG NURJAMAN,1971-08-20,54,L,True,None,58,2029-09-01,...,True,196904101992011001,KASIDI,None,None,SMAN 1 JATILUHUR KABUPATEN PURWAKARTA|SATUAN P...,False,False,True,None


In [18]:
# Database connection configuration to EKinerja 2026
DB_HOST = os.getenv('DB_HOST_2026', 'localhost')
DB_PORT = os.getenv('DB_PORT_2026', '5432')  # PostgreSQL default port
DB_NAME = os.getenv('DB_DATABASE_2026', 'your_database_name')
DB_USER = os.getenv('DB_USERNAME_2026', 'your_username')
DB_PASSWORD = os.getenv('DB_PASSWORD_2026', 'your_password')

# Create connection string for PostgreSQL
connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

print(f"Connecting to database: {DB_NAME} on {DB_HOST}:{DB_PORT}")
print(f"User: {DB_USER}")

# Test connection
try:
    engine_2026 = create_engine(connection_string, echo=False)
    with engine_2026.connect() as connection:
        result = connection.execute(text("SELECT 1 as test"))
        print("✅ Database connection successful!")
        print(f"Connection test result: {result.fetchone()[0]}")
except SQLAlchemyError as e:
    print(f"❌ Database connection failed: {e}")
    print("Please check your database credentials in the .env file")

Connecting to database: erk_ekinerja_2026 on 10.110.32.114:5432
User: postgres
✅ Database connection successful!
Connection test result: 1


In [19]:
# Database connection configuration to ERK
DB_HOST = os.getenv('DB_HOST_ERK', 'localhost')
DB_PORT = os.getenv('DB_PORT_ERK', '5432')  # PostgreSQL default port
DB_NAME = os.getenv('DB_DATABASE_ERK', 'your_database_name')
DB_USER = os.getenv('DB_USERNAME_ERK', 'your_username')
DB_PASSWORD = os.getenv('DB_PASSWORD_ERK', 'your_password')

# Create connection string for PostgreSQL
connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

print(f"Connecting to database: {DB_NAME} on {DB_HOST}:{DB_PORT}")
print(f"User: {DB_USER}")

# Test connection
try:
    engine_erk = create_engine(connection_string, echo=False)
    with engine_erk.connect() as connection:
        result = connection.execute(text("SELECT 1 as test"))
        print("✅ Database connection successful!")
        print(f"Connection test result: {result.fetchone()[0]}")
except SQLAlchemyError as e:
    print(f"❌ Database connection failed: {e}")
    print("Please check your database credentials in the .env file")

Connecting to database: erk_jabar on 10.110.32.115:5432
User: postgres
✅ Database connection successful!
Connection test result: 1


In [20]:
# try:
#     engine_2026.dispose()
# except:
#     pass

# # Recreate the engine
# engine_2026 = create_engine(connection_string, echo=False)

# REVIEW

In [21]:
# Get bulan review
bulan_review = int(str(period)[-2:])

# Query review data
query_review = f"""
SELECT 
    *
FROM 
    reviewer
WHERE 
    bulan = {bulan_review}
"""

df_review = pd.read_sql(query_review, engine_2026)

df_review.head()

,nip,nip_reviewer,hubungan,nomor,bulan,submit,remedial,updated_at,id,created_at
0,198312052022212015,198411092023212019,rekan,1,3,False,False,2026-03-25 00:54:03,4629262,2026-03-25 00:54:03
1,198312052022212015,199110172023212022,rekan,1,3,False,False,2026-03-25 00:54:03,4629264,2026-03-25 00:54:03
2,198312052022212015,199302232022212019,rekan,1,3,False,False,2026-03-25 00:54:03,4629265,2026-03-25 00:54:03
3,198312052022212015,197906282023212003,rekan,1,3,False,False,2026-03-25 00:54:03,4629267,2026-03-25 00:54:03
4,198312052022212015,197510032023211002,rekan,1,3,False,False,2026-03-25 00:54:03,4629268,2026-03-25 00:54:03


## Cleansing Pensiun

In [22]:
# Ambil TMT Pensiun
# Convert period (e.g., 202603) to 2026-03-01, then add one month to get 2026-04-01
tmt_pensiun = pd.to_datetime(str(period) + '01', format='%Y%m%d') + pd.DateOffset(months=1)
tmt_pensiun = tmt_pensiun.strftime('%Y-%m-%d')

# Cleansing pensiun
df_vpd_pensiun = df_vpd.loc[:, ['peg_nip', 'peg_status_kepegawaian_id', 'satuan_kerja_nama', 'unit_kerja_nama', 'peg_tmt_pensiun']]

df_vpd_pensiun = df_vpd_pensiun[df_vpd_pensiun['peg_tmt_pensiun'] == tmt_pensiun]

df_vpd_pensiun.head()

,peg_nip,peg_status_kepegawaian_id,satuan_kerja_nama,unit_kerja_nama,peg_tmt_pensiun


In [23]:
df_direview_pensiun = df_review.groupby('nip').size().reset_index(name='direview_count')


# Check reviewer that are not in df_vpd
df_direview_pensiun = df_direview_pensiun[~df_direview_pensiun['nip'].isin(df_vpd['peg_nip'])]

df_direview_pensiun.head(25)


,nip,direview_count


In [24]:
df_direview_pensiun.count()


nip               0
direview_count    0
dtype: int64

In [25]:
# Update bulan direview dengan delete_placeholder
pensiun_nips = df_direview_pensiun['nip'].tolist()

log_entries = []

for nip in pensiun_nips:
    query = """
    UPDATE reviewer
    SET bulan = :delete_placeholder
    WHERE nip = :nip
    """
    result = execute_update(query, {'nip': nip, 'delete_placeholder': delete_placeholder}, engine_2026, f"Update bulan for {nip}")
    log_entries.append({
        'timestamp': datetime.now(),
        'nip': nip,
        'action': 'update',
        'description': f"Update reviewer bulan {bulan_review} to {delete_placeholder} for {nip}"
    })

# Convert log_entries to DataFrame
df_log = pd.DataFrame(log_entries)

# Create or append log file
now = datetime.now().strftime("%Y%m%d")
# Create log file if it doesn't exist
if not os.path.exists(f'{now}-kuesview-log.csv'):
    df_log.to_csv(f'{now}-kuesview-log.csv', index=False)
else:
    df_log.to_csv(f'{now}-kuesview-log.csv', mode='a', header=False, index=False)

## Kelebihan kocokan

Sudah dihandle dengan generate:remove-kelebihan-kocokan tiap tanggal 28

In [26]:
# Calculate the number of review for each reviewer
df_review_count = df_review.groupby('nip_reviewer').size().reset_index(name='review_count')

# Remove rows where review_count is <= 25
df_review_count_25 = df_review_count[df_review_count['review_count'] > 25]

# Order by review_count in descending order
df_review_count_25 = df_review_count_25.sort_values(by='review_count', ascending=False)

df_review_count_25.head(10)


,nip_reviewer,review_count


In [43]:
# Look for pegawai with no review
df_vpd_nip = df_vpd.loc[:, ['peg_nip', 'peg_nama', 'peg_status_kepegawaian_id', 'satuan_kerja_nama', 'unit_kerja_nama', 'peg_tmt_pensiun', 'inaktif_keterangan', 'inaktif_mulai', 'inaktif_selesai']]

# Remove rows where peg_nip is found on nip_reviewer
df_vpd_nip = df_vpd_nip[~df_vpd_nip['peg_nip'].isin(df_review_count['nip_reviewer'])]

# Remove PPPK PW
df_vpd_nip = df_vpd_nip[df_vpd_nip['peg_status_kepegawaian_id'] != 4]
# Remove gub wagub
df_vpd_nip = df_vpd_nip[~df_vpd_nip['peg_nip'].isin(['3000000002', '3000000001'])]

df_vpd_nip.head()

,peg_nip,peg_nama,peg_status_kepegawaian_id,satuan_kerja_nama,unit_kerja_nama,peg_tmt_pensiun,inaktif_keterangan,inaktif_mulai,inaktif_selesai
226,198311112010012013,DIANA NOVIANTI,1,DINAS PENDIDIKAN,SMAN 1 CIGOMBONG KABUPATEN BOGOR,2043-12-01,Pindahan Dari Luar Pemprov Jabar (Skema Talent...,2026-03-01,2026-04-30
1675,199004262017081002,HERSY APRIAN FITRIADI,1,DINAS PENDIDIKAN,SMAN 2 PADALARANG KABUPATEN BANDUNG BARAT,2050-05-01,Pindahan Dari Luar Pemprov Jabar (Skema Talent...,2026-03-01,2026-04-30
7143,198008312009011002,IWAN SETIAWAN,1,DINAS BINA MARGA DAN PENATAAN RUANG,UPTD PENGELOLAAN JALAN DAN JEMBATAN WILAYAH PE...,2038-09-01,None,None,None
12478,198009292009011004,AEF SAEFULLOH,1,DINAS BINA MARGA DAN PENATAAN RUANG,UPTD PENGELOLAAN JALAN DAN JEMBATAN WILAYAH PE...,2038-10-01,None,None,None
16590,198011102008011004,WANDA JATNIKA,1,DINAS BINA MARGA DAN PENATAAN RUANG,UPTD PENGELOLAAN JALAN DAN JEMBATAN WILAYAH PE...,2038-12-01,None,None,None


In [44]:
# Export to csv
df_vpd_nip.to_csv(f'{period}_nip_no_review.csv', index=False)